# SIDERIUS Quickstart — your first task in one notebook

## 1. What are we building?

A complete, tiny **task package** for SIDERIUS — the framework that runs LLM
agents to propose, implement, tune and interpret ML models against a task YOU
define. The task here is deliberately trivial: **binary classification of
synthetic 4-feature vectors** (256 rows, generated in under a second from a
pinned seed, CPU-only, no downloads). It exists to demonstrate the *workflow
mechanics* — the model does **not** need to train well, and this pack says so
explicitly everywhere.

Two honesty rules govern this notebook:

1. **Every executed cell below shows real output** produced on the landed
   post-PR-12d SIDERIUS source this pack is built against — including the
   launch, artifact-inspection and resume sections (9, 10 and 12), whose
   outputs are verbatim excerpts from the operator-authorized bounded live
   runs of 2026-08-25 (1 iteration × 1 round, `certify_minimal.json`; run
   identity in `PROVENANCE.md`).
2. Nothing is mocked. Where the framework refuses, you will see the refusal.

What a task must provide (the five required manifest sections, from
`workflows/task_composition.py::_REQUIRED_KEYS`): `task_data_path`,
`dataset_profile`, `metric`, `task_health`, `task_config`. Read
`docs/concepts/task-package.md` and `docs/guides/define-a-task.md` alongside
this notebook — they are the framework's own reference for each decision.


## 2. Install and imports

Run this notebook from its own directory (`examples/quickstart/`) inside a
SIDERIUS checkout, with the project virtualenv (`.venv/bin/python` — the
system python is too old). There is nothing to install beyond the repo's own
environment; the quickstart needs only `numpy`, `torch` (CPU), `pyyaml`.

The provenance check matters on machines with several SIDERIUS checkouts: an
editable install in the venv can silently shadow the checkout you are sitting
in, and every result would then describe *another clone*.

> Aside: on a fresh checkout the first framework import emits a `UserWarning`
> about `tidmad_data_config.yaml` falling back to its template. It is harmless
> here — the quickstart never reads TIDMAD's data directory.


In [1]:
import importlib
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # examples/quickstart -> repo root
sys.path.insert(0, str(REPO_ROOT))  # this checkout wins over any installed copy

# Imported via importlib AFTER the sys.path pin above: the pin is the point
# (this checkout must win), so the module load has to come after it.
task_composition = importlib.import_module("workflows.task_composition")

resolved = Path(task_composition.__file__).resolve()
assert resolved.is_relative_to(REPO_ROOT), (
    f"IMPORTING ANOTHER CLONE: {resolved} — fix sys.path before trusting any cell below"
)
print("repo root      :", REPO_ROOT.name)
print("composition at :", resolved.relative_to(REPO_ROOT))
print("python         :", sys.version.split()[0])

repo root      : siderius-arxiv-quickstart
composition at : workflows/task_composition.py
python         : 3.12.13


## 3. Generate tiny data

No data files are committed. The pack ships a **seeded generator**
(`plugins/_quickstart_task.py`) and pins the identity of what it produces in
`declared/data_manifest.json` (seed `20260824`, 4 shards x 64 rows, per-shard
sha256). The materializer regenerates the CSVs bit-identically and **refuses
to write a bundle whose bytes disagree with the committed pins** — a drifted
generator fails here, not three steps later in a run.

The generated CSVs land in a **workspace outside the repository tree**
(`SIDERIUS_QUICKSTART_WORKSPACE`, defaulting under your home). Only DATA is
generated — every declaration, including `task_config.yaml`, is committed
with the pack (section 4 explains the governance rule that makes that
legitimate).


In [2]:
import importlib.util

WORKSPACE = Path(
    os.environ.get(
        "SIDERIUS_QUICKSTART_WORKSPACE", str(Path.home() / "siderius_quickstart_workspace")
    )
)

# Load the pack plugin exactly the way the composition loader executes plugin
# files: by path, under a synthetic module name — never a package import.
_spec = importlib.util.spec_from_file_location(
    "quickstart_task", Path.cwd() / "plugins" / "_quickstart_task.py"
)
qs = importlib.util.module_from_spec(_spec)
sys.modules["quickstart_task"] = qs
_spec.loader.exec_module(qs)

bundle = qs.materialize_run_bundle(Path.cwd(), WORKSPACE)
print("workspace:", WORKSPACE)
for name in sorted(p.name for p in WORKSPACE.iterdir()):
    print("  ", name)
print("\nshards (sha-verified against declared/data_manifest.json):")
for f in sorted(Path(bundle["data_dir"]).iterdir()):
    print("  ", f.name, f.stat().st_size, "bytes")
print("\nfirst 3 rows of", qs.shard_filename(0), ":")
print("\n".join((Path(bundle["data_dir"]) / qs.shard_filename(0)).read_text().splitlines()[:4]))

workspace: /tmp/claude-1004/-home-yuema137-SIDERIUS/f6b9131d-c11a-4396-834a-b89a0cce2662/scratchpad/quickstart_ws_exec
   data

shards (sha-verified against declared/data_manifest.json):
   shard_0000.csv 3542 bytes
   shard_0001.csv 3548 bytes
   shard_0002.csv 3547 bytes
   shard_0003.csv 3555 bytes

first 3 rows of shard_0000.csv :
sample_id,x0,x1,x2,x3,label
s0r000,0.74723715,-0.39100042,0.12363672,-0.51339036,1
s0r001,-0.97176385,-0.89249641,-0.55577463,0.85302526,0
s0r002,0.19307469,0.70307106,-0.73112661,0.79268646,0


## 4. Define the task

A task is **files, not framework edits**. This pack's split:

| file | role | committed? |
|---|---|---|
| `declared/dataset_profile.json` | generic identity (`partition_count`, anchors, health peeks) + **opaque** `topology` the framework never inspects | yes |
| `declared/metric_accuracy.json` | the metric's declaration: id, **direction**, aggregation, scoreability contract | yes |
| `declared/data_manifest.json` | identity pins of the generated data | yes |
| `plugins/_quickstart_task.py` | `TaskDataPath` (4 methods) + optional `TaskScopeCapability` (4 methods) + generator + materializer | yes |
| `plugins/_quickstart_metrics.py` | the metric's *implementation* (an `EvaluationMetric`) | yes |
| `plugins/quickstart_reference_mlp.py` | reference model plugin (`PLUGIN_MODEL_TYPE` contract) | yes |
| `declared/task_config.yaml` | task description + **structured `model_io` forward contract** — what every LLM prompt is told | yes |
| `configs/task_composition/quickstart.yaml` | THE shipped manifest binding all of the above for a run (lives beside `tidmad.yaml`/`pets.yaml`/`davis.yaml`) | yes |

A governance rule worth knowing: guard (a)
(`tests/unit/examples/test_pack_governance.py`) refuses any YAML under
`examples/` with a top-level `task_description`/`forward_contract` key —
EXCEPT a file that a shipped `configs/task_composition/*.yaml` manifest
actually binds (the PR-12d re-scope; pets/davis use exactly this). The
quickstart's `declared/task_config.yaml` is legitimate because
`quickstart.yaml` binds it; an identical file nobody binds is still a
refused parallel copy. **Your own out-of-tree task package has no such
restriction** — commit your `task_config.yaml` wherever your package lives.


In [3]:
print("--- declared/task_config.yaml (committed; model_io is the authority) ---")
print((Path.cwd() / "declared" / "task_config.yaml").read_text())
print("--- configs/task_composition/quickstart.yaml (the SHIPPED manifest) ---")
print((REPO_ROOT / "configs" / "task_composition" / "quickstart.yaml").read_text())

--- declared/task_config.yaml (committed; model_io is the authority) ---
# Quickstart — the pack's OWN task description + forward contract.
#
# Post-PR-12d home (README §7 step 2 executed). Q-12d-1: task-owned
# declarations CO-LOCATE with the pack; `configs/task_composition/
# quickstart.yaml` only POINTS here and carries no second copy of any of it.
# Governance guard (a) exempts exactly this file BECAUSE that shipped
# manifest binds it (`test_pack_governance.py::_composition_bound_task_configs`);
# an identical YAML nobody binds is still a refused parallel copy.
#
# Loaded through the SAME `load_task_config` a legacy run uses, via
# `_compose_task_config` under the composed dataset profile.

task_description: |
  Binary classification of tiny synthetic 4-feature vectors. Each sample is
  x in R^4 drawn uniformly from [-1, 1]; the label is 1 exactly when
  x0*x1 + sin(pi*x2) > x3, with 3% of labels deterministically flipped, so a
  perfect score is impossible by construction. The go

**Why `model_io:` matters.** The typed block is THE authority — `input_shape`,
`output_shape` and `num_classes` are DERIVED from it, and the admission
resource check reads its dtype (without it, a float-native model hits an
int64 fallback — the Pets pack learned this in a real launch). Composing a
non-TIDMAD `model_io` task config only works because PR-12d closed
**F-12d-4** (the class-count cross-check used to read TIDMAD topology
unconditionally and refused every non-TIDMAD profile); the landed behaviour
is pinned by `tests/unit/examples/test_quickstart_pack.py::
test_landed_pin_model_io_task_config_composes_f12d4_fixed`.


## 5. Train/validation/test split

The split is **task-owned vocabulary**: shards 0–1 train, shard 2 validation,
shard 3 final-eval — declared in the profile's opaque `topology.roles` and
enforced by the pack's own scope construction (the framework never learns
what a "shard" is). The optional `TaskScopeCapability` builds scopes; note
the eval scope can never be pointed at a training shard, and a `target`
strategy naming the held-out shard is refused.


In [4]:
from execute_tools.task_data_path import ScopeBuildRequest

impl = qs.QuickstartTaskDataPath()
request = ScopeBuildRequest(round_kind="formal", selection_strategy="snapshot", portion=1.0, seed=7)
train_scope = impl.build_training_scope(request)
eval_scope = impl.build_eval_scope(request)
print(
    "train rows:", len(train_scope.rows), "from shards", sorted({r.shard for r in train_scope.rows})
)
print("eval  rows:", len(eval_scope.rows), "from shard", sorted({r.shard for r in eval_scope.rows}))
balance = {}
for r in train_scope.rows:
    balance[r.label] = balance.get(r.label, 0) + 1
print("train label balance:", dict(sorted(balance.items())))
try:
    impl.build_training_scope(
        ScopeBuildRequest(
            round_kind="formal", selection_strategy="target", portion=1.0, target_partitions=(3,)
        )
    )
except ValueError as refusal:
    print("refused:", refusal)

train rows: 128 from shards [0, 1]
eval  rows: 64 from shard [2]
train label balance: {0: 70, 1: 58}
refused: target partitions [3] are not training shards [0, 1] — the validation/final shards must never leak into a training scope.


## 6. Metrics: primary + direction

A metric is a **declaration** (identity, `direction`, aggregation, an
executable scoreability contract) plus an **implementation** (an
`EvaluationMetric` that may not rewrite the declared id). Get the direction
right — nothing infers it from a name or a sign, and a wrong one silently
inverts an entire campaign.

The scoreability contract runs BEFORE any arithmetic: an unscoreable
deliverable is a *structured refusal* (`NotScoreableResult`), never a
garbage number. On this base the contract vocabulary is closed —
`deliverable_presence` and `tidmad_denoised_h5` (`evaluation_metric.py`);
a custom contract class would be a framework edit, so this pack declares
`deliverable_presence`.

Secondary metrics are optional, **observational only** (they influence no
ordering), and this pack declares none — an absent `secondary_metrics:`
section IS the "task has none" state.


In [5]:
import importlib.util as _ilu

from execute_tools.evaluation_metric import metric_spec_from_declaration

declaration = json.loads((Path.cwd() / "declared" / "metric_accuracy.json").read_text())
spec = metric_spec_from_declaration(declaration)
print("metric id:", spec.id, "| direction:", spec.direction, "| aggregation:", spec.aggregation)

_mspec = _ilu.spec_from_file_location(
    "quickstart_metrics", Path.cwd() / "plugins" / "_quickstart_metrics.py"
)
qm = _ilu.module_from_spec(_mspec)
sys.modules["quickstart_metrics"] = qm
_mspec.loader.exec_module(qm)
metric = qm.QuickstartAccuracyMetric(spec)

probe = WORKSPACE / "metric_probe.json"  # a real file, so presence passes
oracle = {r.sample_id: r.label for r in eval_scope.rows}
probe.write_text(json.dumps(oracle))
perfect = metric.evaluate(
    {0: str(probe)}, evaluation_payload=oracle, task_scope=eval_scope, data_dir=None
)
wrong = metric.evaluate(
    {0: str(probe)},
    evaluation_payload={k: 1 - v for k, v in oracle.items()},
    task_scope=eval_scope,
    data_dir=None,
)
missing = metric.evaluate(
    {0: str(probe) + ".gone"}, evaluation_payload=oracle, task_scope=eval_scope, data_dir=None
)
print("oracle predictions ->", type(perfect).__name__, perfect.scalar)
print("all-wrong          ->", type(wrong).__name__, wrong.scalar)
print(
    "missing deliverable->",
    type(missing).__name__,
    "|",
    missing.verdict.failures[0].requirement,
    "|",
    missing.verdict.failures[0].detail,
)

metric id: accuracy | direction: higher | aggregation: fraction_correct_over_eval_rows
oracle predictions -> MetricResult 1.0
all-wrong          -> MetricResult 0.0
missing deliverable-> NotScoreableResult | completeness | deliverable not found at '/tmp/claude-1004/-home-yuema137-SIDERIUS/f6b9131d-c11a-4396-834a-b89a0cce2662/scratchpad/quickstart_ws_exec/metric_probe.json.gone'


## 7. LLM configuration

Facts from the source (`sdsc_submission_scripts/run_one_iteration.py`):

- **If you specify nothing**, the chain resolves
  `WorkflowLLMConfig.uniform("gemini", "gemini-3.1-pro-preview")` — every
  node on that Gemini model (the reflector defaults to `gemini-2.5-flash`).
  `--llm_model` alone is **deprecated**; nothing fails, but you get a
  provider you may not have keys for.
- **The documented default for real work**: `--llm_config
  llm_configs/openai_tiered_pro.json` (per-node routing; the repository's
  Gate standard uses exactly this file). API keys come from your `.env`.
- `llm_configs/certify_minimal.json` pins every agent role to `gpt-4o-mini`
  — its own header says: plumbing certification only, *not* model
  intelligence. It is the cheapest way to watch the mechanics move.
- In both files the literature-review node routes to `deepseek-v4-pro`
  (visible in the audit below) — one more reason to read the config you
  pass rather than assume a single provider.

This notebook makes **no LLM calls**; the config only matters for the launch
in section 9.


In [6]:
for f in sorted((REPO_ROOT / "llm_configs").glob("*.json")):
    payload = json.loads(f.read_text())
    roles = {k: v for k, v in payload.items() if not k.startswith("_")}

    def _models(node, out):
        if isinstance(node, dict):
            if "model_id" in node:
                out.add(node["model_id"])
            else:
                for v in node.values():
                    _models(v, out)

    models = set()
    _models(roles, models)
    print(f"{f.name:28s} -> {sorted(models)}")

certify_minimal.json         -> ['deepseek-v4-pro', 'gpt-4o-mini']
deepseek_tiered_pro.json     -> ['deepseek-v4-pro']
openai_tiered_pro.json       -> ['deepseek-v4-pro', 'gpt-5.5']
openai_tiered_v1.json        -> ['deepseek-v4-pro', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']


## 8. Create a workspace and compose the run

`compose_run_task_bindings(manifest)` is what the chain launcher runs before
anything else: it resolves every declared family **fail-closed** (a missing
file, a misspelled key, a plugin that raises, an id mismatch — each refuses
by name; nothing ever falls back to TIDMAD) and pins a **semantic
fingerprint** over the resolved values and plugin content hashes. That
fingerprint goes into the workspace's `run_invariants_lock.json`: edit a
plugin and a resume of the old workspace refuses, because it would no longer
be the same scientific run.


In [7]:
from workflows.task_composition import compose_run_task_bindings

SHIPPED_MANIFEST = REPO_ROOT / "configs" / "task_composition" / "quickstart.yaml"
composition = compose_run_task_bindings(str(SHIPPED_MANIFEST))
print("task_data_path_id :", composition.task_data_path_id)
print(
    "metric            :",
    composition.metric.spec.id,
    f"({composition.metric.spec.direction} is better)",
)
print("partitions        :", composition.dataset_profile.partition_count)
print("health family     :", composition.task_health_binding, " (a NAMED absence, never TIDMAD's)")
print(
    "deliverable naming:",
    composition.deliverable_naming.prefix + "_*" + composition.deliverable_naming.extension,
)
print(
    "model plugins     :",
    tuple(composition.model_plugins.required_model_types),
    "from the pack's plugins/ dir",
)
print("semantic fingerprint:", composition.semantic_fingerprint[:16], "…")
for ref in composition.provenance.plugins:
    print("  plugin pinned:", ref.configured_ref.rsplit("/", 1)[-1], ref.content_sha256[:12], "…")

task_data_path_id : quickstart_tabular
metric            : accuracy (higher is better)
partitions        : 4
health family     : explicit_none  (a NAMED absence, never TIDMAD's)
deliverable naming: quickstart_predictions_*.json
model plugins     : ('quickstart_reference_mlp',) from the pack's plugins/ dir
semantic fingerprint: 9645c218f5a84b21 …
  plugin pinned: _quickstart_task.py efe279577fc9 …
  plugin pinned: _quickstart_metrics.py e36fd102dc7e …
  plugin pinned: quickstart_reference_mlp.py ca5544db561c …


A note on **health**: `task_health: {none: true}` is 08b's `EXPLICIT_NONE` —
a *named* absence. Omitting the section entirely would mean "resolve TIDMAD's
health family", which is exactly the silent fallback a composition exists to
prevent. When your real task has collapse modes worth watching, start from
the generic checks (`sample_dispersion_floor`, `categorical_distinct_symbols`,
`categorical_dominant_fraction`) per `docs/guides/define-a-task.md` step 6 —
and note that composed runs currently fire ZERO health gates — the A1
declared debt on the landed PR-12d ledger — so never read a composed run as
health-enforced yet.


## 9. Launch one bounded workflow — executed live 2026-08-25

The command below is the real operator surface (`docs/guides/define-a-task.md`
step 10), bounded to **1 iteration x 1 round**, executed as the
operator-authorized bounded witness with ONE recorded substitution:
`--llm_config llm_configs/certify_minimal.json` (README §7 step 5 — a
bounded witness certifies plumbing, not model intelligence; every role
pinned to gpt-4o-mini). The output below is verbatim excerpts from the
final-witness launch.

Witnessed mechanically, end to end: the SHIPPED manifest composed
fail-closed; the cold-start propose → implement → validate ladder passed a
generated model through the CONTRACT-AWARE validator (the generated-pytest
geometry gap this pack's first live run exposed is fixed in the landed
source — every probe surface of that ladder now renders from the task's
`model_io`); the tuner leg ran 15 planned attempts with typed
structured-failure records; and one attempt performed REAL TRAINING on the
pack's data (cross-entropy, epoch losses below, checkpoint saved) with
both pack plugins loaded inside the training and inference children.

Two honest limits, both visible below. That attempt's inference then
failed at THIS PACK's own deliverable codec — the codec predated the
landed positional-pairing convention (seam C/B7) — batch item A, since
FIXED in this pack with a mutation-proven regression test. The remaining
attempts were refused by the resource probe whenever the planner chose the
legacy focal-loss family, which is geometry-incompatible with a
`[B,2]`-logit contract — batch item B, a FLAGGED framework seam (loss
eligibility is not contract-filtered for composed tasks), outside this
pack. The iteration therefore closed `no_records`, honestly: no scored
result exists and none is claimed.

Also visible: composed-run hygiene ("Reference scores: NOT LOADED — this
run is COMPOSED"), the task-owned scope path ("no SampleSet is built for a
task that declares no physical partition geometry"), and the
TIDMAD-physical GPU probe declaring itself NOT APPLICABLE to a task that
declares no TIDMAD topology. Composed runs still fire **zero health
gates** (the A1 debt), so a passing run makes no health-enforcement claim.

In [12]:
%%bash
# Executed 2026-08-25 as the operator-authorized bounded witness — swapping ONE
# flag per README §7 step 5: --llm_config llm_configs/certify_minimal.json.
# The output below is verbatim excerpts of the final-witness launch log.
cd "$(git rev-parse --show-toplevel)"
WS="${SIDERIUS_QUICKSTART_WORKSPACE:-$HOME/siderius_quickstart_workspace}"

bash sdsc_submission_scripts/run_chain.sh \
    --mode lilab \
    --workspace "$WS/chain" \
    --run_name quickstart_v1 \
    --num_iterations 1 \
    --max_rounds 1 \
    --task_composition configs/task_composition/quickstart.yaml \
    --data_dir "$WS/data" \
    --healthgate_mode blocking \
    --result_authority diagnostic \
    --llm_config llm_configs/openai_tiered_pro.json


Start: workspace does not exist yet — fresh chain at 1
############################################################
  SIDERIUS Iteration Chain — LILAB (foreground)
  Workspace        : /tmp/claude-1004/-home-yuema137-SIDERIUS/f6b9131d-c11a-4396-834a-b89a0cce2662/scratchpad/quickstart_final_ws/chain
  Num iterations   : 1
  [... launch banner trimmed ...]
  Task composition : configs/task_composition/quickstart.yaml
[TOKEN] run_id = quickstart_v1-20260825T055108-1072411
  Hardware      : NVIDIA GeForce RTX 5090 (31.34 GB total, 25.07 GB usable cap), budget=None GB
  [... cold-start interpret; 2 proposal attempts failed codegen (weak certify model); attempt 3 below ...]
    Proposed: simple_mlp_classifier
  [1] Tuning 'simple_mlp_classifier' for 1 rounds...
Reference scores: NOT LOADED — this run is COMPOSED. The legacy TIDMAD reference tables are task-specific science and are never loaded implicitly for a composed run; score-comparison tables are omitted for this run.
  [... attempt 1 s

## 10. Inspect results and artifacts

What a chain workspace holds after a run (all names from the current source):

- `run_invariants_lock.json` — scope, health enablement, effective-config
  sha256 and the **composition fingerprint**; a mismatched resume fails
  closed at startup (that is the system working, not breaking).
- `health_checks_effective.yaml` — the composed health policy actually in
  force, sha-pinned by the lock.
- `{node}_{run_name}.json` — each node's output record (persistence/recovery
  log, never the inter-node channel).
- the deliverables directory — files named by the **declared** template.

The final-witness run produced the chain-level artifacts for this task,
and the first cell below reads them from the REAL workspace (its output is
the witnessed run's): the lock's composition fingerprint is byte-equal to
the one section 8 composed from the SHIPPED manifest, the reference-model
plugin's content sha is pinned, and the lock RECORDS the generated-library
provenance (`{root, source: "env"}`) — the promoted capabilities landed
under that environment-resolved root, never inside the repository
checkout, which stayed byte-identical before/after the run. The
section-12 recovery evidence (manifest replacement + set-aside) was
witnessed on the earlier live-fix workspace; this fresh workspace shows
the plain single-manifest shape. What the workspace does NOT hold — a
scored tuner record and a deliverable file — is exactly what the
section-9 batch blocked. The deliverable/scoring lifecycle itself is
executable NOW through the composed authorities (second cell), which is
exactly what the chain's children do: bind the composition, verify the
binding, persist a deliverable under the declared naming, read it back,
and score it.


In [13]:
import json
import os
from pathlib import Path

ws = Path(
    os.environ.get(
        "SIDERIUS_QUICKSTART_WORKSPACE", str(Path.home() / "siderius_quickstart_workspace")
    )
)
chain = ws / "chain"
if not chain.is_dir():
    print(f"(no chain workspace at {chain} — run section 9 first)")
else:
    lock = json.loads((chain / "run_invariants_lock.json").read_text())
    print("run_invariants_lock.json")
    print("  task_composition_fingerprint :", lock["task_composition_fingerprint"])
    print("  resolved_data_scope          :", lock["resolved_data_scope"])
    for pin in lock["model_plugin_identities"]:
        print(
            f"  model plugin pinned          : {pin['model_type']}  sha256 {pin['content_sha256'][:16]}…"
        )
    print("  generated_library (P1)       :", lock.get("generated_library"))
    print(
        "  health_checks_effective.yaml :",
        "present" if (chain / "health_checks_effective.yaml").is_file() else "MISSING",
    )
    manifest = json.loads((chain / "iter_001" / "manifest.json").read_text())
    print("iter_001/manifest.json")
    print("  status                       :", manifest["status"])
    print(
        "  healthgate_mode / authority  :",
        manifest["healthgate_mode"],
        "/",
        manifest["result_authority"],
    )
    replacement = manifest.get("manifest_replacement")
    if replacement:
        print("  manifest_replacement         :", replacement["replacement_reason"])
    set_aside = sorted(p.name for p in (chain / "iter_001").glob("manifest.replaced.*.json"))
    print("  set-aside prior manifests    :", set_aside or "none")
    print(
        "  task_config_snapshot.yaml    :",
        "present" if (chain / "iter_001" / "task_config_snapshot.yaml").is_file() else "MISSING",
    )

run_invariants_lock.json
  task_composition_fingerprint : 9645c218f5a84b219beb23fba6c9afc2916bf89eea8924c0560e822ce96f5b7b
  resolved_data_scope          : [0, 1, 2, 3]
  model plugin pinned          : quickstart_reference_mlp  sha256 ca5544db561c81c5…
  generated_library (P1)       : {'root': '/tmp/claude-1004/-home-yuema137-SIDERIUS/f6b9131d-c11a-4396-834a-b89a0cce2662/scratchpad/quickstart_final_lib2', 'source': 'env'}
  health_checks_effective.yaml : present
iter_001/manifest.json
  status                       : no_records
  healthgate_mode / authority  : blocking / diagnostic
  set-aside prior manifests    : none
  task_config_snapshot.yaml    : present


In [8]:
from execute_tools.task_data_path import DeliverableWriteRequest, EvaluationReadRequest
from workflows.task_composition import bind_run_task_composition, verify_composition_is_bound

identity = dict(exp_id="exp0", run_name="quickstart_v1", model_type="quickstart_reference_mlp")
deliverable_dir = WORKSPACE / "deliverables"

with bind_run_task_composition(composition, physical_data_root=bundle["data_dir"]):
    verify_composition_is_bound(composition)  # a half-composed run refuses here
    bound_impl = composition.task_data_path  # the COMPOSED instance (its own module load)
    predictions = {r.sample_id: r.label for r in eval_scope.rows}  # stand-in model outputs
    bound_impl.write_deliverable(
        predictions, DeliverableWriteRequest(output_dir=str(deliverable_dir), **identity)
    )
    decoded = bound_impl.read_evaluation_payload(
        EvaluationReadRequest(deliverable_dir=str(deliverable_dir), **identity)
    )
    artifact = deliverable_dir / bound_impl.deliverable_name(**identity)
    outcome = composition.metric.evaluate(
        {0: str(artifact)}, evaluation_payload=decoded, task_scope=eval_scope, data_dir=None
    )

print("deliverable :", artifact.name, f"({artifact.stat().st_size} bytes)")
print("round-trip  :", "exact" if decoded == predictions else "MISMATCH")
print("scored      :", type(outcome).__name__, "=", outcome.scalar)

deliverable : quickstart_predictions_quickstart_reference_mlp_quickstart_v1_exp0_0000.json (962 bytes)
round-trip  : exact
scored      : MetricResult = 1.0


## 11. Inspect the model plugin

In a real chain the implementor agent *writes* model plugins into
`agent_generated/models/` and the validator checks them against the plugin
contract: a module defining `PLUGIN_MODEL_TYPE`, `PLUGIN_CONFIG_CLASS` (a
pydantic config) and `PLUGIN_MODEL_CLASS` (the `nn.Module`). This pack ships
a reference plugin in that exact shape, so the post-12d chain has a model to
start from — and so you can see the contract concretely.


In [9]:
import torch

from execute_tools.task_data_path import EpochSamplingParams

_pspec = importlib.util.spec_from_file_location(
    "quickstart_mlp", Path.cwd() / "plugins" / "quickstart_reference_mlp.py"
)
mlp_mod = importlib.util.module_from_spec(_pspec)
sys.modules["quickstart_mlp"] = mlp_mod
_pspec.loader.exec_module(mlp_mod)

config = mlp_mod.PLUGIN_CONFIG_CLASS()
model = mlp_mod.PLUGIN_MODEL_CLASS(config)
parameters = sum(p.numel() for p in model.parameters())
print("PLUGIN_MODEL_TYPE :", mlp_mod.PLUGIN_MODEL_TYPE)
print("config defaults   :", config.model_dump())
print("parameters        :", parameters)

dataset = impl.training_dataset(
    train_scope, EpochSamplingParams(data_dir=bundle["data_dir"], epoch_seed=0)
)
batch = torch.stack([dataset[i][0] for i in range(8)])
logits = model(batch)
print(
    "forward           :",
    tuple(batch.shape),
    str(batch.dtype),
    "->",
    tuple(logits.shape),
    str(logits.dtype),
)

PLUGIN_MODEL_TYPE : quickstart_reference_mlp
config defaults   : {'segmentation_size': 4, 'batch_size': 32, 'hidden_dim': 16}
parameters        : 114
forward           : (8, 4) torch.float32 -> (8, 2) torch.float32


## 12. Resume (same workspace) — recovery resume executed live

Re-running the section-9 command with the **same `--workspace` and
`--run_name`** resumes: `--auto_resume` is the default (the launcher queries
`scripts/inspect_run_state.py` for the next iteration; `--start_iter N` is
the manual pin). The invariants lock makes resumes honest — same scope, same
health enablement, same effective config, same composition fingerprint, or
the launch refuses at startup. Editing any pack plugin changes the
fingerprint, so a resume of the old workspace fails **closed**; that
refusal is a feature, and the fresh-start section is the answer.

The live witness exercised the RECOVERY case. After the first bounded run
ended `no_records`, re-running the identical command was first REFUSED by
the launcher's stale-fresh guard (real output):

```
Start: auto-resume — inspector computed START_ITER=1
ERROR: Workspace at .../quickstart_live_ws/chain is not empty. Use --force_fresh to clobber existing data, or specify --start_iter N to resume from a specific point.
```

Sharp edge, worth knowing: at **iteration 1** the guard cannot distinguish
"fresh start pointed at the wrong directory" from "recovery of a terminally
`failed`/`no_records` iteration 1" (the #258 case), so the recovery
relaunch needs `--force_fresh` — which on the lilab path deletes NOTHING:
the shell only bypasses the guard, and the launcher then replaces the
terminal manifest through the explicit set-aside + provenance path. From
the witnessed workspace (real artifact):

```
"manifest_replacement": {
  "replacement_reason": "auto_resume recovery: prior iteration manifest was terminal 'no_records' (#258)",
  "previous_manifest_sha256": "c6e5ef80e51a955c…"
}
```

with the prior manifest set aside beside it as
`manifest.replaced.20260825T035133099555Z.json`, the run identity carried
unchanged (`quickstart_v1-20260825T034146-793953`), and the invariants
lock byte-stable across the resume (that workspace's fingerprint,
`ede74e70…`).
A COMPLETED manifest is never replaced by auto-resume, and a fully
complete chain exits at the launcher's idempotency check before any
submission — recovery replacement is scoped to terminal
`failed`/`no_records` slots only.

## 13. Fresh start (new workspace)

A fresh run is a new `--workspace` (and ideally a new `--run_name`). Nothing
in the pack or the repository holds run state — the workspace IS the run.
Delete or abandon the old directory, point the launch at a new one, done.
`--no_auto_resume` starts from iteration 1, and in a non-empty existing
directory the stale-fresh guard then also requires `--force_fresh` (section
12); prefer a new directory, so old evidence stays inspectable.

The materializer is idempotent: re-running section 3 regenerates the same
bytes (the pins guarantee it), so bundles are disposable.


## 14. Replace the example with your own task

Map each quickstart file to your task, in the order
`docs/guides/define-a-task.md` walks them:

1. **Data + profile** — your partitions, your `topology` payload (the
   framework never looks inside), your anchor/health-peek declarations.
2. **`TaskDataPath`** — your four methods; add `TaskScopeCapability` if your
   data is not "N uniform partitions". Exact validation materialization is a
   hard obligation: fail, never pad.
3. **`task_config.yaml`** — your science, with a structured `model_io:`
   block. Commit it with your package; an in-tree pack additionally binds it
   from a shipped `configs/task_composition/*.yaml` (the guard-(a) rule from
   section 4).
4. **Metric declaration + implementation** — one number, one declared
   direction. Everything else is a secondary, and secondaries influence
   nothing.
5. **Health** — `none: true` honestly, or generic checks with thresholds you
   can defend.
6. **Manifest** — point every section at your files (`file:` refs make the
   package relocatable; paths resolve against the manifest's own directory).
   The base composes an out-of-tree package with **zero framework edits** —
   the repository proves that with its `fourth_task` fixture, and this pack
   is the same shape one directory over.
7. **Model plugin** — ship a reference model in the plugin contract and
   declare it in your manifest's `model_plugins:` section (`require:` makes
   an unresolvable plugin a composition refusal, not a mid-run surprise).

Then compose it (section 8) — the fail-closed refusals are your checklist —
and launch bounded (section 9). The pack's landed-boundary pins
(`tests/unit/examples/test_quickstart_pack.py`) keep the composition
behaviours you rely on — section-key refusal, seam-A `config:` delivery,
`model_plugins` require-enforcement, `model_io` composability — executable
evidence rather than prose.
